# Breast Cancer PRS Analysis — Python Component
## Data: Michailidou et al. 2017 (Nature)

This notebook continues the analysis from the R pipeline (scripts/01_gwas_qc_prs.R).

The R script handled:
- Raw GWAS data loading and QC
- Manhattan and QQ plots
- Export of clean SNP table

This notebook handles:
- Visualisation of the GWAS effect size landscape
- PRS simulation using real GWAS effect sizes
- Logistic regression classification of cases vs controls
- ROC curve evaluation

**Why both languages?** R is the standard for GWAS and genomics QC. 
Python is the standard for machine learning. Using both languages will enable effective analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Consistent visual style across all plots
sns.set_theme(style="whitegrid", font_scale=1.2)
BLUE   = "#2E86AB"
PURPLE = "#A23B72"

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load both files exported by the R script
snps = pd.read_csv("../data/processed/prs_snps_clean.csv")
thresholds = pd.read_csv("../data/processed/threshold_sensitivity.csv")

print(f"SNPs loaded: {len(snps):,}")
print(f"Column names: {list(snps.columns)}")
print(f"First 5 rows:")
snps.head()

SNPs loaded: 603,024
Column names: ['snp', 'chr', 'bp37', 'a1', 'a0', 'beta', 'se', 'p', 'eaf']
First 5 rows:


,snp,chr,bp37,a1,a0,beta,se,p,eaf
0,chr10:123337335,10,123337335,G,A,-0.238705,0.009730,6.700000e-133,0.595983
1,chr10:123341525,10,123341525,C,A,0.241006,0.009825,7.300000e-133,0.389552
2,chr10:123340431,10,123340431,G,GC,-0.239180,0.009753,8.200000e-133,0.595421
3,chr10:123337182,10,123337182,C,T,-0.238551,0.009730,9.500000e-133,0.595925
4,chr10:123337117,10,123337117,T,C,-0.238257,0.009730,2.000000e-132,0.595845


In [3]:
print(f"Total SNPs: {len(snps):,}")
print(f"Chromosomes represented: {sorted(snps['chr'].unique())}")
print(f"P-value range: {snps['p'].min():.2e} to {snps['p'].max():.2e}")
print(f"Beta range: {snps['beta'].min():.3f} to {snps['beta'].max():.3f}")

Total SNPs: 603,024
Chromosomes represented: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
P-value range: 6.70e-133 to 4.90e-02
Beta range: -0.366 to 0.423


In [4]:
snps['chr'] = snps['chr'].astype(int)